In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

 

In [ ]:
from  sklearn.metrics import root_mean_squared_log_error as rmsle

In [ ]:
train = pd.read_csv('Dataset/train.csv')
test = pd.read_csv('Dataset/test.csv')
transactions = pd.read_csv('Dataset/transactions.csv')
stores = pd.read_csv('Dataset/stores.csv')
sample_submission = pd.read_csv('Dataset/sample_submission.csv')
oil = pd.read_csv('Dataset/oil.csv')
holidays_events = pd.read_csv('Dataset/holidays_events.csv')

In [ ]:
train_ori = train.copy()
test_ori = test.copy()

In [ ]:
print("train:", train.columns)
print("transaction:", transactions.columns)
print("stores:", stores.columns)
print("holidays event:", holidays_events.columns)
print("oil:", oil.columns)

train: Index(['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion'], dtype='object')
transaction: Index(['date', 'store_nbr', 'transactions'], dtype='object')
stores: Index(['store_nbr', 'city', 'state', 'type', 'cluster'], dtype='object')
holidays event: Index(['date', 'type', 'locale', 'locale_name', 'description', 'transferred'], dtype='object')
oil: Index(['date', 'dcoilwtico'], dtype='object')


## Data checking

In [ ]:
# row and columns
print("row and columns:\n",train.shape)



row and columns:
 (3000888, 6)


In [ ]:
# Info
print("Info:")
train.info()

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000888 entries, 0 to 3000887
Data columns (total 6 columns):
 #   Column       Dtype  
---  ------       -----  
 0   id           int64  
 1   date         object 
 2   store_nbr    int64  
 3   family       object 
 4   sales        float64
 5   onpromotion  int64  
dtypes: float64(1), int64(3), object(2)
memory usage: 137.4+ MB


In [ ]:
# Describe Data
print("\nDescribe Data:")
print(train.describe())


Describe Data:
                 id     store_nbr         sales   onpromotion
count  3.000888e+06  3.000888e+06  3.000888e+06  3.000888e+06
mean   1.500444e+06  2.750000e+01  3.577757e+02  2.602770e+00
std    8.662819e+05  1.558579e+01  1.101998e+03  1.221888e+01
min    0.000000e+00  1.000000e+00  0.000000e+00  0.000000e+00
25%    7.502218e+05  1.400000e+01  0.000000e+00  0.000000e+00
50%    1.500444e+06  2.750000e+01  1.100000e+01  0.000000e+00
75%    2.250665e+06  4.100000e+01  1.958473e+02  0.000000e+00
max    3.000887e+06  5.400000e+01  1.247170e+05  7.410000e+02


In [ ]:
# Head
print("Head:")
print(train.head())

Head:
   id        date  store_nbr      family  sales  onpromotion
0   0  2013-01-01          1  AUTOMOTIVE    0.0            0
1   1  2013-01-01          1   BABY CARE    0.0            0
2   2  2013-01-01          1      BEAUTY    0.0            0
3   3  2013-01-01          1   BEVERAGES    0.0            0
4   4  2013-01-01          1       BOOKS    0.0            0


In [ ]:
# Tail
print("Tail:")
print(train.tail())

Tail:
              id        date  store_nbr                      family     sales  \
3000883  3000883  2017-08-15          9                     POULTRY   438.133   
3000884  3000884  2017-08-15          9              PREPARED FOODS   154.553   
3000885  3000885  2017-08-15          9                     PRODUCE  2419.729   
3000886  3000886  2017-08-15          9  SCHOOL AND OFFICE SUPPLIES   121.000   
3000887  3000887  2017-08-15          9                     SEAFOOD    16.000   

         onpromotion  
3000883            0  
3000884            1  
3000885          148  
3000886            8  
3000887            0  


In [ ]:
# Na check
print("Na check")
print(train.isna().sum())

# Null check
print("Null check")
print(train.isnull().sum())

Na check
id             0
date           0
store_nbr      0
family         0
sales          0
onpromotion    0
dtype: int64
Null check
id             0
date           0
store_nbr      0
family         0
sales          0
onpromotion    0
dtype: int64


In [ ]:
# Duplicated check
print("Duplicated check")
print(train.duplicated().sum())

# Unique check (cat columns)
print("Unique check")
print("family columns", train['family'].nunique())
print("family columns", train['family'].unique())
print("store_nbr columns:", train['store_nbr'].nunique())

Duplicated check
0
Unique check
family columns 33
family columns ['AUTOMOTIVE' 'BABY CARE' 'BEAUTY' 'BEVERAGES' 'BOOKS' 'BREAD/BAKERY'
 'CELEBRATION' 'CLEANING' 'DAIRY' 'DELI' 'EGGS' 'FROZEN FOODS' 'GROCERY I'
 'GROCERY II' 'HARDWARE' 'HOME AND KITCHEN I' 'HOME AND KITCHEN II'
 'HOME APPLIANCES' 'HOME CARE' 'LADIESWEAR' 'LAWN AND GARDEN' 'LINGERIE'
 'LIQUOR,WINE,BEER' 'MAGAZINES' 'MEATS' 'PERSONAL CARE' 'PET SUPPLIES'
 'PLAYERS AND ELECTRONICS' 'POULTRY' 'PREPARED FOODS' 'PRODUCE'
 'SCHOOL AND OFFICE SUPPLIES' 'SEAFOOD']
store_nbr columns: 54


In [ ]:
print("store_nbr di stores columns:", stores['store_nbr'].nunique())

store_nbr di stores columns: 54


## Menggabungkan data train, stores, transactions

In [ ]:
train_merged = pd.merge(train, stores, on='store_nbr', how='left')

print("Bentuk data setelah digabung dengan stores:")
print(train_merged.head())

train_final = pd.merge(train_merged, transactions, on=['date', 'store_nbr'], how='left')

train_final['transactions'] = train_final['transactions'].fillna(0)

print("\nBentuk data final setelah digabung dengan transactions:")
train = train_final
print(train.head())

Bentuk data setelah digabung dengan stores:
   id        date  store_nbr      family  sales  onpromotion   city  \
0   0  2013-01-01          1  AUTOMOTIVE    0.0            0  Quito   
1   1  2013-01-01          1   BABY CARE    0.0            0  Quito   
2   2  2013-01-01          1      BEAUTY    0.0            0  Quito   
3   3  2013-01-01          1   BEVERAGES    0.0            0  Quito   
4   4  2013-01-01          1       BOOKS    0.0            0  Quito   

       state type  cluster  
0  Pichincha    D       13  
1  Pichincha    D       13  
2  Pichincha    D       13  
3  Pichincha    D       13  
4  Pichincha    D       13  

Bentuk data final setelah digabung dengan transactions:
   id        date  store_nbr      family  sales  onpromotion   city  \
0   0  2013-01-01          1  AUTOMOTIVE    0.0            0  Quito   
1   1  2013-01-01          1   BABY CARE    0.0            0  Quito   
2   2  2013-01-01          1      BEAUTY    0.0            0  Quito   
3   3  2013-01

In [ ]:
# oil['dcoilwtico'] = oil['dcoilwtico'].interpolate(method='linear').bfill()

# oil.head()

In [ ]:
# train.info()
# print(' ')
# train.head(20)

In [ ]:
# train['date'] = pd.to_datetime(train['date'])

# train = train.set_index('date').drop(columns='id')

# train.head(20)

In [ ]:
# from sklearn.preprocessing import LabelEncoder

# # Inisialisasi LabelEncoder
# le = LabelEncoder()

# # Fit dan transform kolom family, lalu tambah 1 agar mulai dari 1
# train['family'] = le.fit_transform(train['family']) 

# # Cek hasilnya
# print(train['family'].unique())

In [ ]:
# train.head(20)

In [ ]:
# train['month'] = train.index.month
# train['day_of_week'] = train.index.dayofweek
# train['year'] = train.index.year
# train['is_weekend'] = (train.index.dayofweek >= 5).astype(int)

In [ ]:
# train.head(20)

In [ ]:
# # Contoh membuat lag 1 hari
# train['sales_lag_1'] = train.groupby(['store_nbr', 'family'])['sales'].shift(1)

# # Contoh membuat rata-rata bergerak 7 hari
# train['rolling_mean_7'] = train.groupby(['store_nbr', 'family'])['sales'].transform(lambda x: x.shift(1).rolling(window=7).mean())

In [ ]:
# train.head()